### Optuna
- 최적의 파라미터를 찾기 위한 라이브러리
- create_study() 라는 내장 함수를 이용하여 특정 모델의 최적의 파라미터를 서치
- GridSearchCV에 비해서 속도 면에서 우세
    - GridSearchCV는 파라미터의 모든 조합의 fit하고 검증의 결과를 확인
    - Optuna는 확률 기반으로 모든 조합을 활용하지 않는다.
- 조합의 수 차이
    - GridSearchCV : params로 제어
    - Optuna : n_trials 매개변수로 제어

In [ ]:
#라이브러리 설치
# !pip install optuna

In [3]:
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer

In [6]:
#iris 데이터 로드
X, y = load_iris(return_X_y=True)

In [7]:
#objective 함수를 생성 : 파라미터의 조합, 파이프, 폴드, 검증 방법
def objective(trial):
    #SVC 모델의 파라미터 조합을 생성
    #suggest_XXXX
        #suggest_int, suggest_float : 정수, 실수 형태의 파라미터 조합 (시작값, 종료값, log 매개변수)
            #log 매개변수 : False가 기본값, True로 변경하면 로그 스케일로 조합을 생성 (실수 형태에서 사용)
            #suggest_categorical : 특정 범주 조합
    C = trial.suggest_float('C', 1e-3, 10.0, log=True)
    gamma=trial.suggest_float('gamma', 1e-4, 1.0, log=True)
    kernel=trial.suggest_categorical('kernel', ['linear','rbf'])
    model=SVC(C=C, gamma=gamma, kernel=kernel)

    pipe=Pipeline(
        [
            ('std', StandardScaler()),
            ('clf', model)
        ]
    )

    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores=cross_val_score(pipe, X, y, cv=cv, scoring=make_scorer(f1_score, average='macro'))

    return scores.mean()

In [ ]:
study=optuna.create_study(
    direction='maximize',
    study_name='class_ml_tuning',
)
study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=True
)

In [9]:
#최적의 파라미터 값을 출력
study.best_params

{'C': 0.8529559643844574, 'gamma': 0.00011380771778917206, 'kernel': 'linear'}

In [10]:
#최적의 스코어 확인
study.best_value

0.9732664995822891